# AndinaLog 03B | Productos | Tratamiento v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
import numpy as np

ENTORNO="auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE="/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO_REQUERIDA="GIAD-M3-S4-PRODUCTOS-diagnostico-didactico-v2"
VERSION_TRATAMIENTO="GIAD-M3-S4-PRODUCTOS-tratamiento-didactico-v2"
MIN_LOTES_RESPALDO=20
COLUMNAS_BRONZE=["producto_id","nombre_producto","categoria_logistica",
    "temperatura_conservacion_requerida_c","tolerancia_temperatura_c",
    "precio_unitario_bob","costo_unitario_bob"]
# Patrones consistentes observados en este dataset, no especificaciones universales.
PATRONES={"Fresco":{"objetivo_c":4.0,"tolerancia_c":2.0,"vida_dias":18},
          "Congelado":{"objetivo_c":-18.0,"tolerancia_c":2.0,"vida_dias":180},
          "Seco":{"objetivo_c":20.0,"tolerancia_c":5.0,"vida_dias":365}}

def encontrar_raiz():
    if ENTORNO=="drive" or (ENTORNO=="auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz=Path(RUTA_PROYECTO_DRIVE)
        if not (raiz/"proyecto-integrador/01_diagnostico/andinalog_productos/salidas/andinalog_productos_didactico_v2_diagnosticado.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(),*Path.cwd().parents]:
        if (carpeta/"proyecto-integrador/01_diagnostico/andinalog_productos/salidas/andinalog_productos_didactico_v2_diagnosticado.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ=encontrar_raiz()
RUTA_BRONZE=RAIZ/"datasets/AndinaLog_03B_Bronce/andinalog_productos.csv"
RUTA_INVENTARIO=RAIZ/"datasets/AndinaLog_03B_Bronce/andinalog_inventory_tracking.csv"
ENTRADA=RAIZ/"proyecto-integrador/01_diagnostico/andinalog_productos/salidas/andinalog_productos_didactico_v2_diagnosticado.csv"
SALIDAS=RAIZ/"proyecto-integrador/02_tratamiento/andinalog_productos/salidas"
df=pd.read_csv(ENTRADA,dtype="string",encoding="utf-8-sig",keep_default_na=False)
bronze=pd.read_csv(RUTA_BRONZE,dtype="string",encoding="utf-8-sig",keep_default_na=False)
requeridas=["fila_bronze","en_cuarentena",*COLUMNAS_BRONZE]
requeridas += [f"{c}_{s}" for c in COLUMNAS_BRONZE for s in ("en_cuarentena","motivo")]
faltan=sorted(set(requeridas)-set(df.columns))
if faltan: raise ValueError(f"Faltan columnas del diagnóstico: {faltan}")
if list(bronze.columns)!=COLUMNAS_BRONZE: raise ValueError("Esquema Bronze inesperado")
if len(df)!=len(bronze) or df["fila_bronze"].duplicated().any():
    raise ValueError("Filas diagnosticadas incompletas o duplicadas")
if not df["fila_bronze"].eq(pd.Series(range(1,len(df)+1),dtype="string")).all():
    raise ValueError("Orden Bronze inesperado")
pd.testing.assert_frame_equal(df[COLUMNAS_BRONZE],bronze)
huella=hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
original=df.copy(deep=True)
df=df.rename(columns={"en_cuarentena":"en_cuarentena_diagnostico"})
print("Entrada:",len(df),"| Cuarentena diagnóstica:",int(df["en_cuarentena_diagnostico"].eq("True").sum()))


## 1. Vida útil observada en inventario

La referencia se deriva de fechas **observadas** en el Bronze de inventario; no se usan vencimientos imputados por otro notebook. Se excluyen copias exactas y claves en conflicto. Solo se acepta una duración si hay al menos 20 lotes del producto con fechas válidas y una sola duración observada. El hash del inventario permite identificar esta referencia auxiliar.


In [ ]:
HASH_INVENTARIO=hashlib.sha256(RUTA_INVENTARIO.read_bytes()).hexdigest()
inv=pd.read_csv(RUTA_INVENTARIO,dtype="string",encoding="utf-8-sig",keep_default_na=False)
columnas_inv=["movimiento_id","lote_id","producto_id","centro_distribucion",
    "fecha_ingreso","fecha_salida","fecha_vencimiento","cantidad_ingreso",
    "cantidad_salida","cantidad_merma","dias_en_almacen","costo_unitario_bob"]
if list(inv.columns)!=columnas_inv: raise ValueError("Esquema de inventario inesperado")
inv_producto=inv["producto_id"].str.strip().str.upper()
inv_ingreso=pd.to_datetime(inv["fecha_ingreso"],format="%Y-%m-%d",errors="coerce")
inv_vencimiento=pd.to_datetime(inv["fecha_vencimiento"],format="%Y-%m-%d",errors="coerce")
vida=(inv_vencimiento-inv_ingreso).dt.days
firma_inv=pd.util.hash_pandas_object(inv[columnas_inv],index=False)
copia_inv=inv.duplicated(columnas_inv,keep="first")
conflicto_inv=pd.Series(False,index=inv.index)
for clave in ["movimiento_id","lote_id"]:
    k=inv[clave].str.strip()
    variantes=firma_inv.groupby(k,dropna=False).transform("nunique")
    conflicto_inv |= k.ne("") & k.duplicated(keep=False) & variantes.gt(1)
base_inv=(~copia_inv & ~conflicto_inv & inv_producto.str.fullmatch(r"PROD-\d{3}").fillna(False) &
          inv_ingreso.notna() & inv_vencimiento.notna() & vida.gt(0))
historico=pd.DataFrame({"producto_id":inv_producto.loc[base_inv],"vida_dias":vida.loc[base_inv]})
resumen_vida=historico.groupby("producto_id")["vida_dias"].agg(["count","nunique","first"])
vida_aprobada=resumen_vida.loc[(resumen_vida["count"]>=MIN_LOTES_RESPALDO) &
                                resumen_vida["nunique"].eq(1),"first"]
df["producto_id_tratado"]=df["producto_id"].str.strip().str.upper()
df["vida_util_observada_dias"]=pd.to_numeric(df["producto_id_tratado"].map(vida_aprobada),errors="coerce")
df["lotes_respaldo_vida_util"]=pd.to_numeric(df["producto_id_tratado"].map(resumen_vida["count"]),errors="coerce")
print("Productos con vida útil única y respaldo:",len(vida_aprobada))


## 2. Valores preparados e imputaciones visibles

La categoría inválida se infiere solo si objetivo, tolerancia y vida útil coinciden con un único patrón. La temperatura faltante se imputa solo si categoría reconocida, tolerancia y vida útil también coinciden. Ambas decisiones se registran por variable, y el Bronze queda intacto.


In [ ]:
df["acciones_tratamiento"]=""
df["motivos_tratamiento"]=""
def anotar(mascara,accion,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    for c,texto in [("acciones_tratamiento",accion),("motivos_tratamiento",motivo)]:
        previo=df.loc[mascara,c]
        df.loc[mascara,c]=previo.where(previo.eq(""),previo+" | ")+texto

df["producto_id_normalizado"]=df["producto_id_tratado"].ne(df["producto_id"])
anotar(df["producto_id_normalizado"],"NORMALIZAR_PRODUCTO_ID","Espacios y mayúsculas normalizados")
df["categoria_logistica_tratada"]=df["categoria_logistica"]
df["temperatura_conservacion_requerida_c_tratada"]=pd.to_numeric(
    df["temperatura_conservacion_requerida_c"].str.strip(),errors="coerce")
df["tolerancia_temperatura_c_tratada"]=pd.to_numeric(df["tolerancia_temperatura_c"].str.strip(),errors="coerce")
df["precio_unitario_bob_tratado"]=pd.to_numeric(df["precio_unitario_bob"].str.strip(),errors="coerce")
df["costo_unitario_bob_tratado"]=pd.to_numeric(df["costo_unitario_bob"].str.strip(),errors="coerce")

categoria_inferida=pd.Series("",index=df.index,dtype="string")
for cat,pat in PATRONES.items():
    coincide=(df["temperatura_conservacion_requerida_c_tratada"].sub(pat["objetivo_c"]).abs().le(0.01) &
              df["tolerancia_temperatura_c_tratada"].sub(pat["tolerancia_c"]).abs().le(0.01) &
              df["vida_util_observada_dias"].eq(pat["vida_dias"]))
    categoria_inferida= categoria_inferida.mask(coincide,cat)
df["categoria_imputada"]=(~df["categoria_logistica"].isin(PATRONES) & categoria_inferida.ne(""))
df.loc[df["categoria_imputada"],"categoria_logistica_tratada"]=categoria_inferida.loc[df["categoria_imputada"]]
anotar(df["categoria_imputada"],"IMPUTAR_CATEGORIA",
       "Categoría inferida por objetivo, tolerancia y vida útil observada del producto")

df["temperatura_imputada"]=False
for cat,pat in PATRONES.items():
    elegible=(df["temperatura_conservacion_requerida_c"].str.strip().eq("") &
      df["categoria_logistica_tratada"].eq(cat) &
      df["tolerancia_temperatura_c_tratada"].sub(pat["tolerancia_c"]).abs().le(0.01) &
      df["vida_util_observada_dias"].eq(pat["vida_dias"]))
    df.loc[elegible,"temperatura_conservacion_requerida_c_tratada"]=pat["objetivo_c"]
    df.loc[elegible,"temperatura_imputada"]=True
anotar(df["temperatura_imputada"],"IMPUTAR_TEMPERATURA_OBJETIVO",
       "Objetivo estimado por categoría, tolerancia y vida útil observada del producto")


## 3. Duplicados después de preparar y decisión final

Se compara la fila completa preparada. La primera aparición se conserva; las copias posteriores se excluyen. Si el mismo ID preparado mantiene atributos diferentes, todas sus variantes quedan en cuarentena. El precio menor al costo es una advertencia comercial, no una corrección automática.


In [ ]:
columnas_preparadas=["producto_id_tratado","nombre_producto","categoria_logistica_tratada",
    "temperatura_conservacion_requerida_c_tratada","tolerancia_temperatura_c_tratada",
    "precio_unitario_bob_tratado","costo_unitario_bob_tratado"]
firma=pd.util.hash_pandas_object(df[columnas_preparadas],index=False)
variantes=firma.groupby(df["producto_id_tratado"],dropna=False).transform("nunique")
conflicto=(df["producto_id_tratado"].ne("") &
          df["producto_id_tratado"].duplicated(keep=False) & variantes.gt(1))
copia=df.duplicated(columnas_preparadas,keep="first") & ~conflicto
anotar(copia,"EXCLUIR_COPIA","Copia posterior equivalente después de preparar el producto")
anotar(conflicto,"CUARENTENA_CONFLICTO","Mismo producto_id tratado con atributos contradictorios")

df["motivo_cuarentena_final"]=""
def cuarentena_si(mascara,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    previo=df.loc[mascara,"motivo_cuarentena_final"]
    df.loc[mascara,"motivo_cuarentena_final"]=previo.where(previo.eq(""),previo+" | ")+motivo

cuarentena_si(copia,"Copia excluida del Silver")
cuarentena_si(conflicto,"ID de producto con atributos contradictorios")
cuarentena_si(~df["producto_id_tratado"].str.fullmatch(r"PROD-\d{3}").fillna(False),"ID inválido")
cuarentena_si(df["nombre_producto"].str.strip().eq(""),"Nombre faltante")
cuarentena_si(~df["categoria_logistica_tratada"].isin(PATRONES),"Categoría sin resolución confiable")
cuarentena_si(df["temperatura_conservacion_requerida_c_tratada"].isna(),"Objetivo térmico faltante o inválido")
cuarentena_si(df["tolerancia_temperatura_c_tratada"].isna() |
             df["tolerancia_temperatura_c_tratada"].le(0),"Tolerancia inválida")
cuarentena_si(df["precio_unitario_bob_tratado"].isna() |
             df["precio_unitario_bob_tratado"].le(0),"Precio inválido")
cuarentena_si(df["costo_unitario_bob_tratado"].isna() |
             df["costo_unitario_bob_tratado"].le(0),"Costo inválido")
for cat,pat in PATRONES.items():
    es=df["categoria_logistica_tratada"].eq(cat)
    cuarentena_si(es & df["temperatura_conservacion_requerida_c_tratada"].notna() &
      df["temperatura_conservacion_requerida_c_tratada"].sub(pat["objetivo_c"]).abs().gt(0.01),
      "Objetivo térmico incoherente con categoría tratada")
    cuarentena_si(es & df["tolerancia_temperatura_c_tratada"].notna() &
      df["tolerancia_temperatura_c_tratada"].sub(pat["tolerancia_c"]).abs().gt(0.01),
      "Tolerancia incoherente con categoría tratada")

df["precio_menor_costo"]=(df["precio_unitario_bob_tratado"]<df["costo_unitario_bob_tratado"])
anotar(df["precio_menor_costo"],"REVISAR_MARGEN","Precio por debajo del costo declarado")
df["en_cuarentena_final"]=df["motivo_cuarentena_final"].ne("")
df["decision_tratamiento"]=np.where(df["en_cuarentena_final"],"CUARENTENA","SILVER")
df["apto_umbral_termico_observado"]=(~df["en_cuarentena_final"] &
    ~df["categoria_imputada"] & ~df["temperatura_imputada"])
silver=df.loc[~df["en_cuarentena_final"]].copy()
cuarentena_final=df.loc[df["en_cuarentena_final"]].copy()


## 4. Comprobaciones y exportación

Silver y cuarentena final forman una partición del diagnóstico. El maestro Silver tiene un ID tratado único. Las inferencias se conservan como tales para que el análisis de IoT sepa qué umbrales fueron observados y cuáles fueron estimados.


In [ ]:
pd.testing.assert_frame_equal(df[[c for c in original.columns if c!="en_cuarentena"]],original[[c for c in original.columns if c!="en_cuarentena"]])
assert len(df)==len(silver)+len(cuarentena_final)
assert silver["producto_id_tratado"].is_unique
assert silver["motivo_cuarentena_final"].eq("").all()
assert cuarentena_final["motivo_cuarentena_final"].ne("").all()
assert silver["categoria_logistica_tratada"].isin(PATRONES).all()
assert silver["temperatura_conservacion_requerida_c_tratada"].notna().all()
assert not (silver["apto_umbral_termico_observado"] &
            (silver["categoria_imputada"] | silver["temperatura_imputada"])).any()
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()==huella
assert hashlib.sha256(RUTA_INVENTARIO.read_bytes()).hexdigest()==HASH_INVENTARIO

SALIDAS.mkdir(parents=True,exist_ok=True)
ruta_silver=SALIDAS/"andinalog_productos_didactico_v2_silver.csv"
ruta_cuarentena=SALIDAS/"andinalog_productos_didactico_v2_cuarentena_final.csv"
silver.to_csv(ruta_silver,index=False,encoding="utf-8-sig")
cuarentena_final.to_csv(ruta_cuarentena,index=False,encoding="utf-8-sig")
print("Entrada:",len(df),"| Silver:",len(silver),"| Cuarentena final:",len(cuarentena_final))
print("IDs normalizados:",int(df["producto_id_normalizado"].sum()),
      "| Categorías imputadas:",int(df["categoria_imputada"].sum()),
      "| Temperaturas imputadas:",int(df["temperatura_imputada"].sum()),
      "| Copias excluidas:",int(copia.sum()))
print("Silver:",ruta_silver)
print("Cuarentena:",ruta_cuarentena)
display(df[["fila_bronze","producto_id","producto_id_tratado","categoria_logistica",
            "categoria_logistica_tratada","categoria_imputada","temperatura_imputada",
            "decision_tratamiento","motivo_cuarentena_final"]].tail(12))

# Third treatment output and published distinction between prior and final quarantine.
assert "en_cuarentena_diagnostico" in df and "en_cuarentena" not in df
assert len(silver)+len(cuarentena_final)==len(original)
metricas={"filas_entrada":len(df),"filas_silver":len(silver),
          "filas_cuarentena_final":len(cuarentena_final),
          "filas_recuperadas":int((df["en_cuarentena_diagnostico"].eq("True") & ~df["en_cuarentena_final"]).sum())}
for campo in ["categoria_imputada","temperatura_imputada","vencimiento_imputado", "capacidad_revisar_ficha"]:
    if campo in df: metricas[campo]=int(df[campo].fillna(False).astype(bool).sum())
reporte_calidad=pd.DataFrame([{"metrica":k,"valor":v} for k,v in metricas.items()])
reporte_calidad.to_csv(SALIDAS/("andinalog_productos_didactico_v2_reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(metricas)
